In [14]:
import pandas as pd
import os
import sys
from pathlib import Path
from sqlalchemy import create_engine,text
from dotenv import load_dotenv

load_dotenv()

True

In [15]:
ROOT = Path().resolve().parents[1] 
sys.path.append(str(ROOT))
print(ROOT)

C:\Users\ArneBogaert\Desktop\HoGent\2eBachelor\Sem2\Data Engineering Project\ProjectRepo\data-engineering-project-i-2025-2026-g09-1


In [16]:
df_locations = pd.read_excel(f"{ROOT}/data/zipcodes_num_nl_2025.xls")
df_locations.head()

,Postcode,Plaatsnaam,Deelgemeente,Hoofdgemeente,Provincie
0,612,Sinterklaas,NaN,Sinterklaas,NaN
1,1000,Brussel,Neen,BRUSSEL,BRUSSEL
2,1005,Verenigde Vergadering van de Gemeenschappelijke,NaN,Verenigde Vergadering van de Gemeenschappelijke,NaN
3,1006,Raad van de Vlaamse Gemeenschapscommissie,NaN,Raad van de Vlaamse Gemeenschapscommissie,NaN
4,1007,Assemblée de la Commission Communautaire Franç...,NaN,Assemblée de la Commission Communautaire Franç...,NaN


In [17]:
df_locations.drop(columns=["Deelgemeente"], inplace=True)
df_locations = df_locations.iloc[1:]

In [18]:
df_locations.rename(columns={
        "Postcode" : "PostalCode",
        "Plaatsnaam" : "Municipality",
        "Hoofdgemeente" : "MainMunicipality",
        "Provincie" : "Province"
    }, inplace=True)

In [19]:
## Add locationKey (incremented)
df_locations["LocationKey"] = range(1,len(df_locations) + 1)
df_locations.head()

,PostalCode,Municipality,MainMunicipality,Province,LocationKey
1,1000,Brussel,BRUSSEL,BRUSSEL,1
2,1005,Verenigde Vergadering van de Gemeenschappelijke,Verenigde Vergadering van de Gemeenschappelijke,NaN,2
3,1006,Raad van de Vlaamse Gemeenschapscommissie,Raad van de Vlaamse Gemeenschapscommissie,NaN,3
4,1007,Assemblée de la Commission Communautaire Franç...,Assemblée de la Commission Communautaire Franç...,NaN,4
5,1008,Kamer van Volksvertegenwoordigers,Kamer van Volksvertegenwoordigers,NaN,5


In [20]:
df_locations[df_locations["PostalCode"] == 9000]

,PostalCode,Municipality,MainMunicipality,Province,LocationKey
2462,9000,Gent,GENT,OOST-VLAANDEREN,2462


In [21]:
df_locations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2763 entries, 1 to 2763
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   PostalCode        2763 non-null   int64 
 1   Municipality      2763 non-null   object
 2   MainMunicipality  2763 non-null   object
 3   Province          2720 non-null   object
 4   LocationKey       2763 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 108.1+ KB


In [22]:
## alles in title casing zetten (Camel Casing) buiten postalCode en LocationKey want dat is een nummer daarom [1:-1]
for i in df_locations.columns[1:-1]:
        df_locations[i] = df_locations[i].str.title()
        
df_locations.head()

,PostalCode,Municipality,MainMunicipality,Province,LocationKey
1,1000,Brussel,Brussel,Brussel,1
2,1005,Verenigde Vergadering Van De Gemeenschappelijke,Verenigde Vergadering Van De Gemeenschappelijke,NaN,2
3,1006,Raad Van De Vlaamse Gemeenschapscommissie,Raad Van De Vlaamse Gemeenschapscommissie,NaN,3
4,1007,Assemblée De La Commission Communautaire Franç...,Assemblée De La Commission Communautaire Franç...,NaN,4
5,1008,Kamer Van Volksvertegenwoordigers,Kamer Van Volksvertegenwoordigers,NaN,5


## Functie om LocationKey toe te voegen aan Dataframe

In [23]:
def getLocationKey(postalCode=None, Province=None, Municipality=None, MainMunicipality=None, server="Local"):
    conn_str = os.getenv("db_lokaal_conn" if server == 'Local' else "df_VIC_conn") 
    engine = create_engine(conn_str)
    
    # data ophalen, de where1=1 is zodat we AND's kunnen toevoegen want soms hebben we niet alle data zoals postcode,municipality,...
    query = "SELECT LocationKey FROM DimLocation WHERE 1=1"
    params = {} 

    # Helper functie om te checken of een waarde niet None en niet NaN is
    def is_valid(val):
        return val is not None and not pd.isna(val)

    # Check elke parameter en voeg toe aan de query als deze valide is
    if is_valid(postalCode):
        query += " AND PostalCode = :postalCode"
        params['postalCode'] = postalCode
    
    if is_valid(Province):
        query += " AND Province = :Province"
        params['Province'] = Province
        
    if is_valid(Municipality):
        query += " AND Municipality = :Municipality"
        params['Municipality'] = Municipality
        
    if is_valid(MainMunicipality):
        query += " AND MainMunicipality = :MainMunicipality"
        params['MainMunicipality'] = MainMunicipality

    # Uitvoeren met de engine
    with engine.connect() as connection:
        df_result = pd.read_sql(sql=text(query), con=connection, params=params)
    
    # Veiligheidscheck: bestaat de resultaatset en is deze niet leeg?
    if not df_result.empty:
        return int(df_result["LocationKey"][0])
    else:
        return None
    
    

    

In [24]:
# test
getLocationKey(postalCode=9000, Municipality='Gent', MainMunicipality='Gent', Province='Oost-Vlaanderen')